In [4]:
# =========================
# 1. Imports
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [5]:
# =========================
# 2. Load Data
# =========================
df = pd.read_csv("../laptop_price.csv", encoding="latin1")

df.head()


,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


In [6]:
# =========================
# 3. Basic Cleaning
# =========================

# RAM: "8GB" -> 8
df["Ram"] = df["Ram"].str.replace("GB", "").astype(int)

# Weight: "1.37kg" -> 1.37
df["Weight"] = df["Weight"].str.replace("kg", "").astype(float)

# CPU frequency extraction (GHz)
df["Cpu_GHz"] = (
    df["Cpu"]
    .str.extract(r'(\d+\.\d+|\d+)GHz')
    .astype(float)
)

# Screen resolution: extract width and height
res = df["ScreenResolution"].str.extract(r'(\d+)x(\d+)')
df["ScreenWidth"] = res[0].astype(int)
df["ScreenHeight"] = res[1].astype(int)

# Drop text-heavy columns we already extracted info from
df = df.drop(columns=["Cpu", "ScreenResolution", "Product"])


In [7]:
# =========================
# 4. Define Features & Target
# =========================
TARGET = "Price_euros"

NUM_FEATURES = [
    "Inches",
    "Ram",
    "Weight",
    "Cpu_GHz",
    "ScreenWidth",
    "ScreenHeight"
]

CAT_FEATURES = [
    "Company",
    "TypeName",
    "Memory",
    "Gpu",
    "OpSys"
]

X = df[NUM_FEATURES + CAT_FEATURES]
y = df[TARGET]


In [8]:
# =========================
# 5. Train / Validation / Test Split
# =========================
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)


In [9]:
# =========================
# 6. Preprocessing Pipeline
# =========================
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUM_FEATURES),
        ("cat", categorical_transformer, CAT_FEATURES)
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)
X_val_prep   = preprocessor.transform(X_val)
X_test_prep  = preprocessor.transform(X_test)


In [10]:
# =========================
# 7. FCNN Regression Model
# =========================
model = Sequential([
    Dense(256, activation="relu", input_shape=(X_train_prep.shape[1],)),
    Dropout(0.3),

    Dense(128, activation="relu"),
    Dropout(0.2),

    Dense(64, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()


c:\Users\Eyll\Codes\PythonCodes\CSE421HW3\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        43,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 84,737 (331.00 KB)

 Trainable params: 84,737 (331.00 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# =========================
# 8. Train Model
# =========================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train_prep, y_train,
    validation_data=(X_val_prep, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1691370.1250 - mae: 1106.6212 - val_loss: 2263859.7500 - val_mae: 1273.6317
Epoch 2/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1563696.3750 - mae: 1053.7451 - val_loss: 1837239.5000 - val_mae: 1120.2256
Epoch 3/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 845149.1875 - mae: 688.3703 - val_loss: 429064.1562 - val_mae: 444.5356
Epoch 4/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 226391.4062 - mae: 356.3062 - val_loss: 198259.2188 - val_mae: 303.8059
Epoch 5/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 156448.3438 - mae: 271.9860 - val_loss: 161757.7500 - val_mae: 271.2047
Epoch 6/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 134423.1562 - mae: 249.8013 - val_loss: 146710.9844 - val_mae: 254.3845
Epoch 7/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 130567.5859 - mae: 246.4045 - val_loss: 139534.2812 - val_mae: 242.6962
Epoch 8/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 109441.1641 - ma

In [12]:
# =========================
# 9. Evaluate Model
# =========================
y_pred = model.predict(X_test_prep).flatten()

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f} €")
print(f"RMSE : {rmse:.2f} €")
print(f"R²   : {r2:.3f}")


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
MAE  : 180.09 €
RMSE : 268.52 €
R²   : 0.816


In [13]:
# =========================
# 10. Sample Predictions
# =========================
pd.DataFrame({
    "Actual_Price": y_test.values[:10],
    "Predicted_Price": y_pred[:10]
})


,Actual_Price,Predicted_Price
0,349.0,414.100098
1,739.0,827.836060
2,959.0,1120.418213
3,1769.0,1759.051392
4,269.0,463.015411
5,529.0,469.772797
6,1026.0,1124.872925
7,2449.0,2194.045654
8,597.0,979.963928
9,2729.0,2556.073730
